# U.S. Gun Violence Trends

An exploratory analysis of U.S. gun incident records from 2014 to 2022. It looks at how recorded incidents vary over time and across states and cities.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

sns.set_theme(style='whitegrid')

## Load and prepare the data

The analysis uses incident records and population estimates for state and city comparisons.

### Incident records

In [ ]:
data_raw = pd.read_csv('data/gun_incidents.csv', parse_dates=['date'], index_col=0)
data_raw.head(3)

### City population estimates (2015)

In [ ]:
city_population = pd.read_csv('data/city_population_2015.csv')
city_population.head()

### State population estimates (2022)

In [ ]:
population = pd.read_excel(
    'data/state_population_2022.xlsx',
    sheet_name='Data',
    names=['state', 'population'],
)
population.head()

## Prepare the incident records

### Incident records

In [ ]:
data_ts = data_raw.copy()
data_ts.sort_values('date', inplace=True)

In [ ]:
data_ts = data_ts.astype({
    'state': 'category',
    'city': 'category',
    'n_injured': 'uint8',
    'n_killed': 'uint8',
})

In [ ]:
data_ts['week_day'] = data_ts['date'].dt.weekday
data_ts['month_day'] = data_ts['date'].dt.day
data_ts['month'] = data_ts['date'].dt.month
data_ts['year'] = data_ts['date'].dt.year
data_ts['n_victims'] = data_ts['n_killed'] + data_ts['n_injured']

In [ ]:
data_ts.isnull().sum()

In [ ]:
data_ts['year'].value_counts()

In [ ]:
# 2013 has very incomplete data, so we'll drop it as it won't be useful for our analysis
data_ts = data_ts[data_ts['year'] != 2013]

### State names for mapping

In [ ]:
data_geo = data_ts.copy()
data_geo = data_geo.astype({'city': 'str', 'state': 'str'})

state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'District of Columbia': 'DC', 'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI',
    'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME',
    'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
    'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE',
    'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM',
    'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI',
    'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX',
    'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
}
data_geo['state_code'] = data_geo['state'].map(state_to_code)

### State population data

In [ ]:
population = population.iloc[4:, -2:].reset_index(drop=True)
population = population.dropna(subset=['state', 'population']).copy()
population['state'] = population['state'].astype(str).str.strip()
state_population = population

### Incident rates per resident

In [ ]:
victims_per_state = data_ts.groupby('state', observed=True).agg({
    'n_killed': 'sum',
    'n_injured': 'sum',
    'n_victims': 'sum',
}).rename(columns={
    'n_killed': 'Total Killed',
    'n_injured': 'Total Injured',
    'n_victims': 'Total Victims',
})
victims_per_state['Number of Incidents'] = data_ts.groupby('state', observed=True).size()
victims_per_state = victims_per_state.reset_index()

In [ ]:
data_demo_states = victims_per_state.merge(state_population, on='state', how='inner')

columns_to_transform = ['Number of Incidents', 'Total Killed', 'Total Injured', 'Total Victims']
data_per_capita = data_demo_states.copy()
data_per_capita[columns_to_transform] = data_per_capita[columns_to_transform].div(
    data_per_capita['population'], axis=0
)
data_per_capita.rename(columns={
    'Number of Incidents': 'Incidents per 1M residents',
    'Total Killed': 'People Killed per 1M residents',
    'Total Injured': 'People Injured per 1M residents',
    'Total Victims': 'People Affected per 1M residents',
}, inplace=True)
data_per_capita['state_code'] = data_per_capita['state'].map(state_to_code)
data_per_capita['Incidents per 1M residents'] = data_per_capita['Incidents per 1M residents'].round(2)

## Incident trends

### Incidents by year

In [ ]:
# plot the number of incidents per year
plt.figure(figsize=(15, 5))
sns.countplot(x='year', data=data_ts)
plt.title('Number of incidents per year')
plt.xlabel('Year')
plt.ylabel('Number of incidents')
plt.show()

The 2022 records cover only part of the year, so that total is not directly comparable with full years.

### Incidents by month

In [ ]:
# group by year and month and sum the incidents
# needs ".size()" before aggregating
grouped_data = data_ts.groupby(['year', 'month']).size().reset_index(name='incidents')

# pivot the data to get months as rows and years as columns
pivot_data = grouped_data.pivot(index='month', columns='year', values='incidents')

palette = sns.color_palette('tab10', n_colors=len(pivot_data.columns))
plt.figure(figsize=(15, 5))
sns.lineplot(data=pivot_data, palette='tab10', dashes=False, linewidth=1.5)

plt.title('Sum of Incidents per Month for Each Year (2014-2022)')
plt.xlabel('Month')
plt.ylabel('Sum of Incidents')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))

plt.show()



Monthly counts show how recorded incidents are distributed across the calendar year. These are counts rather than rates, and 2022 is incomplete.

### Incidents by day of the week

In [ ]:
plt.figure(figsize=(15, 5))

sns.countplot(data=data_ts, x='week_day')
plt.title('Number of Incidents per Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Number of Incidents')
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.show()

Weekend totals are higher in this dataset. This is a descriptive comparison of recorded counts and does not explain why the pattern occurs.

### Calendar dates with the most recorded incidents

In [ ]:
# find the dates with the most incidents accross all years
top_dates = data_ts.groupby(['month_day', 'month']).size().reset_index(name='incidents').sort_values('incidents', ascending=False).reset_index(drop=True)
top_dates.head(10)

This ranking combines the same month and day across years. It does not adjust for incomplete years or changes in reporting.

### Average interval between recorded incidents by state

In [ ]:
data_mean_time = data_ts.sort_values(by=['state', 'date'])
column_name = 'Mean time between shootings'
data_mean_time[column_name] = data_mean_time.groupby('state')['date'].diff()
mean_time_per_state = pd.DataFrame(data_mean_time.groupby('state')[column_name].mean())
mean_time_per_state.sort_values(by=column_name, inplace=True)
mean_time_per_state[column_name] = mean_time_per_state[column_name].apply(lambda x: f'{round(x.seconds/3600)} hours' if x.days==0 else f'{x.days} days')
mean_time_per_state.head(5)

In [ ]:
mean_time_per_state.tail(5)

These averages describe gaps between recorded incidents in this dataset. They depend on the coverage and reporting period.

### Weekly deaths and injuries

In [ ]:
daily_data = data_ts.groupby('date').agg({'n_killed': 'sum',
                                          'n_injured': 'sum'}).reset_index()

daily_data.set_index('date', inplace=True)
weekly_data = daily_data.resample('W').sum()

fig = px.line(weekly_data, x=weekly_data.index , y=['n_killed', 'n_injured'], title='Weekly Time Series', line_shape='linear')
fig.show()

The chart sums recorded deaths and injuries by week. It shows totals, not population-adjusted rates.

## State and city comparisons

The state view compares raw totals with totals divided by population. The city view focuses on 2015, when matching city population estimates are available.

Start with the total number of recorded incidents by state.

In [ ]:
data_geo.reset_index(inplace=True)
incident_counts = data_geo.groupby(["state"])['incident_id'].nunique().reset_index(name='incident_count')
incident_counts['state_code'] = incident_counts['state'].apply(lambda x : state_to_code[x])
incident_counts.sort_values(by='incident_count', ascending=False).head(5)

Raw counts are difficult to compare across states with different population sizes.

In [ ]:
incidents_map = px.choropleth(
    incident_counts,  
    locations='state_code',  
    color='incident_count', 
    hover_name='state',  
    locationmode='USA-states',  
    color_continuous_scale=[(0, 'rgb(255,204,204)'), (0.5, 'rgb(255,102,102)'), (1, 'rgb(255,0,0)')], #so that the color scale is red
    scope="usa", 
    title='Recorded incidents by state (2014–2022)',
    labels={'incident_count':'Gun Violence Incidents'}  # Label for the color scale
)
incidents_map.show()

The next map divides the recorded incidents over the full study period by 2022 state population estimates. The result is cumulative, not an annual rate.

In [ ]:
min_color = data_per_capita['Incidents per 1M residents'].min()
max_color = data_per_capita['Incidents per 1M residents'].max() / 2
inc_pc_map = px.choropleth(
    data_per_capita,
    locations='state_code',
    color='Incidents per 1M residents',
    hover_name='state',
    locationmode='USA-states',
    color_continuous_scale='Blues',
    scope='usa',
    title='Recorded incidents per 1M residents (2014–2022)',
    range_color=(min_color, max_color),
)
inc_pc_map.show()

In [ ]:
data_per_capita[['state', 'Incidents per 1M residents']].sort_values(by='Incidents per 1M residents', ascending=False).head(3)

In [ ]:
data_per_capita[['state', 'Incidents per 1M residents']].sort_values(by='Incidents per 1M residents', ascending=False).tail(3)

The map shows cumulative incident counts per million residents, using 2022 population estimates as the denominator.

District of Columbia is a federal district rather than a state, and its dense urban area makes a direct state comparison difficult. Read its rate separately.

### City rates in 2015

In [ ]:
data_ts_2015 = data_ts[data_ts['year'] == 2015]
city_incidents = (
    data_ts_2015.groupby(['city', 'state'], observed=True)
    .size()
    .reset_index(name='incident_count')
)
city_incidents['city'] = city_incidents['city'].astype(str)
city_incidents['state'] = city_incidents['state'].astype(str)
city_incidents.sort_values('incident_count', ascending=False, inplace=True)
top_15_cities = city_incidents.head(15)

In [ ]:
top_cities_pop = top_15_cities.merge(city_population, on=['city', 'state'], how='inner')
top_cities_pop['incident_count_per_100k'] = (
    top_cities_pop['incident_count'] / top_cities_pop['population'] * 100_000
)
top_cities_pop['city_label'] = top_cities_pop['city'] + ', ' + top_cities_pop['state']
top_cities_pop.sort_values('incident_count_per_100k', ascending=False, inplace=True)

In [ ]:
plt.figure(figsize=(15, 5))
sns.barplot(data=top_cities_pop, x='city_label', y='incident_count_per_100k')
plt.title('Incidents per 100,000 residents in 15 high-count cities (2015)')
plt.xlabel('City')
plt.ylabel('Incidents per 100,000 residents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

This chart shows population-adjusted counts among the 15 cities with the highest raw incident totals in 2015. It is not a ranking of every U.S. city.